In [41]:
import math
import os
import numpy as np
import pandas as pd

df_pings_gps_original = pd.read_csv("saida/linha_913_final.csv", parse_dates=["datahora"])
df_paradas = pd.read_csv("dados brutos/paradas_913_unificado.csv")

df_pings_gps_original.head(3)

,ordem,latitude,longitude,datahora,linha,temp_max,temp_min,chuva,hora,dia_semana,fim_semana,feriado,dia_util
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913,38.4,36.2,0.0,13,1,0,0,1
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913,38.5,34.7,0.0,13,1,0,0,1
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913,38.5,34.7,0.0,13,1,0,0,1


Versão com numba

In [ ]:
import math
import numpy as np
import pandas as pd
from numba import njit, float64
from numba.typed import Dict
from numba import types

# == Configurações ======================
RAIO_SNAP_M = 150
TIMEOUT_SEC = 30 * 60
MAX_PULOS   = 3

# == Haversine vetorizado com numba ======================
# Distância de um ponto a um array de pontos, em metros.
@njit(cache=True)
def haversine_vec(lat, lon, lats, lons):
    R = 6_371_000.0
    n = len(lats)
    dists = np.empty(n)
    for i in range(n):
        dlat = math.radians(lats[i] - lat)
        dlon = math.radians(lons[i] - lon)
        a = (math.sin(dlat / 2) ** 2
             + math.cos(math.radians(lat))
             * math.cos(math.radians(lats[i]))
             * math.sin(dlon / 2) ** 2)
        dists[i] = R * 2 * math.asin(math.sqrt(a))
    return dists

@njit(cache=True)
def snap_idx_contextual(lat, lon, stop_lats, stop_lons, stop_seqs, raio, seq_minima):
    dists = haversine_vec(lat, lon, stop_lats, stop_lons)
    best_idx = -1
    best_dist = 1e18
    for i in range(len(dists)):
        if dists[i] <= raio and stop_seqs[i] > seq_minima:
            if dists[i] < best_dist:
                best_dist = dists[i]
                best_idx = i
    return best_idx

# Retorna índice da parada mais próxima ou -1 se além do raio.
@njit(cache=True)
def snap_mais_proximo(lat, lon, stop_lats, stop_lons, raio):
    """Snap sem restrição — retorna o mais próximo dentro do raio."""
    dists = haversine_vec(lat, lon, stop_lats, stop_lons)
    idx = np.argmin(dists)
    return int(idx) if dists[idx] <= raio else -1

# == Loop principal com numba ======================
@njit(cache=True)
def processar_veiculo(
    pings_lat, pings_lon, pings_ts,
    stop_lats, stop_lons, stop_seqs, stop_dirs,
    raio, timeout_sec, max_pulos):

    idx_a_list     = []
    idx_b_list     = []
    t_saida_list   = []
    t_chegada_list = []
    pulos_list     = []

    # Contadores de descarte
    descartes_mesma_parada = 0
    descartes_max_pulos    = 0

    estado_idx = -1
    estado_ts  = 0.0

    for i in range(len(pings_lat)):

        if estado_idx == -1:
            s = snap_mais_proximo(pings_lat[i], pings_lon[i], stop_lats, stop_lons, raio)
            if s != -1:
                estado_idx = s
                estado_ts  = pings_ts[i]
            continue

        if pings_ts[i] - estado_ts > timeout_sec:
            s = snap_mais_proximo(pings_lat[i], pings_lon[i], stop_lats, stop_lons, raio)
            if s != -1:
                estado_idx = s
                estado_ts  = pings_ts[i]
            continue

        seq_minima = stop_seqs[estado_idx]
        s = snap_idx_contextual(
            pings_lat[i], pings_lon[i],
            stop_lats, stop_lons, stop_seqs,
            raio, seq_minima
        )

        if s == -1:
            # Meramente estatístico: saber se o descarte foi por estar fora de qualquer parada ou por estar na mesma parada.
            s_livre = snap_mais_proximo(pings_lat[i], pings_lon[i], stop_lats, stop_lons, raio)
            if s_livre == estado_idx:
                descartes_mesma_parada += 1
            continue

        if s == estado_idx:
            descartes_mesma_parada += 1
            continue

        dir_atual = stop_dirs[estado_idx]
        seq_atual = stop_seqs[estado_idx]
        dir_nova  = stop_dirs[s]
        seq_nova  = stop_seqs[s]

        if dir_nova != dir_atual:
            continue
        if seq_nova <= seq_atual:
            continue

        pulos = seq_nova - seq_atual - 1
        if pulos > max_pulos:
            descartes_max_pulos += 1
            continue

        idx_a_list.append(estado_idx)
        idx_b_list.append(s)
        t_saida_list.append(estado_ts)
        t_chegada_list.append(pings_ts[i])
        pulos_list.append(pulos)

        estado_idx = s
        estado_ts  = pings_ts[i]

    return idx_a_list, idx_b_list, t_saida_list, t_chegada_list, pulos_list, descartes_mesma_parada, descartes_max_pulos

# == Arrays de paradas ======================
stop_lats = df_paradas["stop_lat"].to_numpy(dtype=np.float64)
stop_lons = df_paradas["stop_lon"].to_numpy(dtype=np.float64)
stop_seqs = df_paradas["stop_sequence"].to_numpy(dtype=np.int64)
stop_dirs = df_paradas["direction_id"].to_numpy(dtype=np.int64)

# Roda uma vez com dado mínimo para compilar função numba antes do loop
var_a_ser_descartada = processar_veiculo(
    np.array([0.0]), np.array([0.0]), np.array([0.0]),
    stop_lats, stop_lons, stop_seqs, stop_dirs,
    RAIO_SNAP_M, TIMEOUT_SEC, MAX_PULOS
)

# ===================================================
# Processamento por veículo
# ===================================================
df_gps = df_pings_gps_original.sort_values(["ordem", "datahora"]).copy()
df_gps["ts"] = df_gps["datahora"].astype(np.int64) / 1e9

# Colunas de contexto que devem ser copiadas
ctx_cols = ["hora", "dia_semana", "fim_semana", "feriado", "dia_util", "chuva", "temp_max"]

trechos = []
total_mesma_parada = 0
total_max_pulos    = 0

for veiculo, grupo in df_gps.groupby("ordem"):
    pings_lat = grupo["latitude"].to_numpy(dtype=np.float64)
    pings_lon = grupo["longitude"].to_numpy(dtype=np.float64)
    pings_ts  = grupo["ts"].to_numpy(dtype=np.float64)
    ctx       = grupo[ctx_cols].to_numpy()

    idx_a, idx_b, t_saida, t_chegada, pulos, d_mesma, d_pulos = processar_veiculo(
        pings_lat, pings_lon, pings_ts,
        stop_lats, stop_lons, stop_seqs, stop_dirs,
        float(RAIO_SNAP_M), float(TIMEOUT_SEC), float(MAX_PULOS)
    )

    total_mesma_parada += d_mesma
    total_max_pulos    += d_pulos

    for k in range(len(idx_a)):
        # Ping de chegada: achar posição pelo timestamp
        ping_chegada_pos = np.searchsorted(pings_ts, t_chegada[k])

        trechos.append({
            "veiculo":      veiculo,
            "direction_id": int(stop_dirs[idx_a[k]]),
            "stop_a":       df_paradas.iloc[idx_a[k]]["stop_id"],
            "stop_b":       df_paradas.iloc[idx_b[k]]["stop_id"],
            "nome_a":       df_paradas.iloc[idx_a[k]]["stop_name"],
            "nome_b":       df_paradas.iloc[idx_b[k]]["stop_name"],
            "t_saida":      pd.Timestamp(t_saida[k],  unit="s"),
            "t_chegada":    pd.Timestamp(t_chegada[k], unit="s"),
            "segundos":     t_chegada[k] - t_saida[k],
            "pulos":        int(pulos[k]),
            **dict(zip(ctx_cols, ctx[min(ping_chegada_pos, len(ctx)-1)]))
        })

# Limpamos trechos com duração menor que 30 segundos ou maior que 30 minutos, são provavelmente erro.
df_trechos = pd.DataFrame(trechos)
df_trechos = df_trechos[
    (df_trechos["segundos"] >= 30) &
    (df_trechos["segundos"] <= 1800)
].reset_index(drop=True)


print(df_trechos.shape)
print(df_trechos["pulos"].value_counts().sort_index())

total_trechos = len(trechos)
print(f"Trechos válidos: {total_trechos}")
print(f"Descarte mesma parada: {total_mesma_parada} -> {100 * total_mesma_parada / (total_trechos + total_mesma_parada + total_max_pulos):.1f}%")
print(f"Descarte max pulos: {total_max_pulos} -> {100 * total_max_pulos / (total_trechos + total_mesma_parada + total_max_pulos):.1f}%")

(133772, 17)
pulos
0    115206
1     11569
2      5778
3      1219
Name: count, dtype: int64
Trechos válidos: 134067
Descarte mesma parada: 220382 -> 59.8%
Descarte max pulos: 13999 -> 3.8%


In [46]:
df_trechos.head(20)

,veiculo,direction_id,stop_a,stop_b,nome_a,nome_b,t_saida,t_chegada,segundos,pulos,hora,dia_semana,fim_semana,feriado,dia_util,chuva,temp_max
0,B10001,0,3053O00028C2,3053O00002C9,Ponto Final: Del Castilho :: Linhas para Zona ...,Metrô Del Castilho / Shopping Nova América,2025-12-16 14:58:38,2025-12-16 15:09:56,678.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
1,B10001,0,3053O00002C9,3050O00014C0,Metrô Del Castilho / Shopping Nova América,Democráticos - Saída 7,2025-12-16 15:09:56,2025-12-16 15:14:04,248.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
2,B10001,0,3050O00014C0,3157O00014C0,Democráticos - Saída 7,Vila do João,2025-12-16 15:14:04,2025-12-16 15:17:40,216.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
3,B10001,0,3157O00014C0,3157O00019C0,Vila do João,Clínica da Família Adib Jatene,2025-12-16 15:17:40,2025-12-16 15:18:41,61.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
4,B10001,0,3157O00019C0,3105O00020C0,Clínica da Família Adib Jatene,CCMN - Geografia,2025-12-16 15:18:41,2025-12-16 15:21:16,155.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
5,B10001,0,3105O00020C0,3105O00021C0,CCMN - Geografia,CENPES - Geografia,2025-12-16 15:21:16,2025-12-16 15:21:47,31.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
6,B10001,0,3105O00021C0,3105O00010C0,CENPES - Geografia,CT - CCMN,2025-12-16 15:21:47,2025-12-16 15:22:49,62.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
7,B10001,0,3105O00010C0,3105O00031C0,CT - CCMN,CT - Letras,2025-12-16 15:22:49,2025-12-16 15:24:52,123.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1
8,B10001,0,3105O00031C0,3105O00015C0,CT - Letras,Belas Artes - Reitoria,2025-12-16 15:24:52,2025-12-16 15:26:25,93.0,3,15.0,1.0,0.0,0.0,1.0,0.0,35.1
9,B10001,0,3105O00015C0,3105O00016C0,Belas Artes - Reitoria,CT - Letras,2025-12-16 15:26:25,2025-12-16 15:28:59,154.0,0,15.0,1.0,0.0,0.0,1.0,0.0,35.1


In [44]:
df_trechos.to_csv("saida/trechos_913.csv", index=False)